In [ ]:
## Written by Erol Balkovic
## EPFL 5/5/2026
## erol.balkovic@epfl.ch

In [ ]:
from simsopt.configs.zoo import get_data
from simsopt.mhd import Vmec

import numpy as np
import matplotlib.pyplot as plt

from spectre import SPECTRE, SPECTREout
from spectre.file_io import write_input_parameters_to_toml
from spectre.converters import simsvmec2spectre
from spectre.converters import (fix2free,
    params_to_simsopt_boundary,
    plot_freeboundary_modes,
)

%load_ext autoreload
%autoreload 2
#%matplotlib ipympl

In [ ]:
## Load W7-X coils from simsopt zoo
_, _, _, _, bs_obj = get_data("w7x")
print(f"Number of coils (full torus): {len(bs_obj.coils)}")

In [ ]:
## Build fixed-boundary SPECTRE input from W7-X VMEC

vmec = Vmec("input.W7-X_standard_configuration")
params = simsvmec2spectre(vmec, mpol=9, ntor=8, interface_guess='map2disc', tfluxes=[0.2, 1.])

print(f"nvol={params.physics.nvol}, mpol={params.physics.mpol}, ntor={params.physics.ntor}")
print(f"lfreebound={params.physics.lfreebound}, lconstraint={params.physics.lconstraint}")

In [ ]:
## Build plasma boundary from InputParameters, then extend to computational boundary

plasma_boundary = params_to_simsopt_boundary(params, which="plasma")
intermediate_comp_boundary = plasma_boundary.copy()
intermediate_comp_boundary.extend_via_normal(-0.2)   # negative → outward

comp_boundary = intermediate_comp_boundary#.condense_spectrum(Fourier_continuation=False)[0]


# Plot original vs extended cross-sections
nfp = plasma_boundary.nfp
phis = [0.0, 1 / (4 * nfp), 1 / (2 * nfp)]
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for phi, ax in zip(phis, axes):
    for surf, label in [(plasma_boundary, "plasma"), (comp_boundary, "comp")]:
        cs = surf.cross_section(phi=phi, thetas=200)
        R = np.sqrt(cs[:, 0] ** 2 + cs[:, 1] ** 2)
        ax.plot(R, cs[:, 2], label=label)
    ax.set_aspect("equal")
    ax.set_title(f"φ = {phi:.4f}")
    ax.legend()
fig.tight_layout()

In [ ]:
## Convert to free boundary and write TOML

result = fix2free(
    params, bs_obj, comp_boundary,
    filename='w7x_freeb.toml', vacuum=True
)

write_input_parameters_to_toml(result, 'w7x_vacuum.toml')
nvol = result.physics.nvol
print(f"lfreebound : {result.physics.lfreebound}")
print(f"nvol       : {nvol}")
print(f"curpol     : {result.physics.curpol:.4g} T·m")
print(f"phiedge    : {result.physics.phiedge:.4g} Wb")
print(f"tflux[nvol-1:nvol+1] : {result.physics.tflux[nvol - 1 : nvol + 1]}")


fig = plot_freeboundary_modes(bs_obj, result, log_scale=False)  # linear scale: see which modes dominate
plt.show()

In [ ]:
## Run SPECTRE, trace field lines, and plot Poincaré section. This is not a converged equilibrium, but we can look at the unconverged state. 

spectre = SPECTRE.from_input_params(result, name="w7x_freeb")
spectre.run()

# force SPECTRE to evaluate the field by calling these functions: 
from spectre import get_xinit_specwrap,get_fmn_force_mask
xin = get_xinit_specwrap(spectre)
mask = get_fmn_force_mask(spectre)

out = spectre.create_spectreout()

# Initial points: evenly spaced in s ∈ (-1, 1), θ = 0, one array per volume
num_trajectories = 8
num_transits = 5
mvol = out.input.physics.Mvol

init_pts = [
    np.array([[x, 0.0] for x in np.linspace(-.95, .95, 2)]),
    np.array([[x, 0.0] for x in np.linspace(-.95, .95, 15)]), 
    np.array([[x, 0.0] for x in np.linspace(-.95, .95, 15)]), 
]

out.trace_fieldlines(init_pts, num_phi_planes=32, num_transits=num_transits)

fig, ax = plt.subplots(figsize=(8, 7))
out.tracer.plot(ax=ax)
out.plot_infaces(ax=ax)
ax.set_aspect("equal")
ax.set_title("W7-X Poincaré section (ζ = 0)")
plt.tight_layout()
plt.show()